# Robust and Lightweight Detection of AI-Generated Text
**Hybrid stylometric–perplexity–transformer fusion with a normalization defense, evaluated on RAID**

Author: Amit Katoch, SCSE, IILM University, Greater Noida

### How to run (Google Colab)
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. At the end, download `results.zip` (tables as CSV/LaTeX + 300-dpi figures).

Approximate run time on a free T4: **~60–90 min** with default settings
(set `CFG["run_logo"] = False` for a ~30-min first pass).

### Experiments
| ID | Question | Output |
|----|----------|--------|
| E1 | How well does each detector work in-distribution? | Table 1, per-generator / per-domain breakdown |
| E2 | How robust is each detector to 8 character/word-level attacks, with and without the normalization defense? | Table 2, Fig. 2 |
| E3 | Do detectors generalise to an **unseen LLM family** (leave-one-family-out)? | Table 3, Fig. 3 |
| E4 | What does each detector cost (parameters, latency)? | Table 4 |

Detectors: (D1) char n-gram TF-IDF + LR, (D2) stylometric features + gradient boosting,
(D3) perplexity features from a small LM (Pythia-160M) + LR, (D4) fine-tuned DistilRoBERTa,
(**Proposed**) Unicode normalization → late fusion of D1–D4 via a logistic meta-learner.

In [ ]:
# !pip -q install -U transformers accelerate scikit-learn pandas matplotlib seaborn nltk
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs], check=False)
try:
    import transformers, seaborn, nltk  # noqa
except ImportError:
    _pip("transformers", "accelerate", "scikit-learn", "pandas", "matplotlib", "seaborn", "nltk")

In [ ]:
import os, re, gc, json, math, time, random, unicodedata, zipfile
import numpy as np
import pandas as pd

CFG = dict(
    seed=42,
    raid_url="https://dataset.raid-bench.xyz/train_none.csv",   # RAID train split, no adversarial attacks
    raid_path="/content/raid_train_none.csv",
    per_model_n=1500,      # machine-generated docs sampled per generator (11 generators)
    human_n=15000,         # human docs sampled (spread across domains)
    min_chars=200, max_chars=2000,
    run_tag="v2_flat",     # used in the results zip name
    flatten_layout=True,   # collapse line breaks: RAID human texts have far fewer breaks (shortcut), see Sec. 4
    split=(0.7, 0.1, 0.2), # train / val (for fusion + thresholds) / test
    tf_model="distilroberta-base",
    ppl_model="EleutherAI/pythia-160m",   # scoring LM; not one of RAID's generators
    max_len=256, epochs=2, batch=32, lr=3e-5,
    ppl_batch=16,
    attack_n=2000,         # machine test docs attacked per attack type
    run_logo=True, logo_epochs=1,
    run_paraphrase=False, paraphrase_n=300,   # optional T5 paraphrase attack (slow)
    out_dir="/content/results" if os.path.isdir("/content") else "results",
)
os.makedirs(CFG["out_dir"], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass
set_seed(CFG["seed"])

FAMILY = {"gpt2": "openai", "gpt3": "openai", "gpt4": "openai", "chatgpt": "openai",
          "mistral": "mistral", "mistral-chat": "mistral",
          "mpt": "mpt", "mpt-chat": "mpt",
          "llama-chat": "llama",
          "cohere": "cohere", "cohere-chat": "cohere",
          "human": "human"}

## 1. Data: stratified sample of RAID (no-attack split)
We stream the CSV in chunks and keep a uniform random sample per (generator, domain) cell,
so the sample is balanced across the 11 generators and 8 domains.

In [ ]:
USECOLS = ["id", "model", "decoding", "repetition_penalty", "attack", "domain", "generation"]

def download_raid(cfg):
    if os.path.exists(cfg["raid_path"]) and os.path.getsize(cfg["raid_path"]) > 1e6:
        return cfg["raid_path"]
    print("Downloading RAID (train, no attacks)… this takes a few minutes")
    r = subprocess.run(["wget", "-q", "-O", cfg["raid_path"], cfg["raid_url"]])
    if r.returncode != 0 or os.path.getsize(cfg["raid_path"]) < 1e6:
        raise RuntimeError("Download failed; see load_raid_hf() fallback below")
    return cfg["raid_path"]

def stratified_sample_csv(path, cfg, n_domains=8, chunksize=200_000):
    """Uniform sample per (model, domain) cell using random keys (one pass, bounded memory)."""
    rng = np.random.default_rng(cfg["seed"])
    k_machine = math.ceil(cfg["per_model_n"] / n_domains)
    k_human = math.ceil(cfg["human_n"] / n_domains)
    keep = None
    for chunk in pd.read_csv(path, usecols=USECOLS, chunksize=chunksize):
        chunk = chunk[(chunk["attack"].fillna("none") == "none") & chunk["generation"].notna()]
        chunk = chunk[chunk["generation"].str.len() >= cfg["min_chars"]]
        chunk = chunk.assign(_u=rng.random(len(chunk)))
        keep = chunk if keep is None else pd.concat([keep, chunk], ignore_index=True)
        k = np.where(keep["model"] == "human", k_human, k_machine)
        keep["_rank"] = keep.groupby(["model", "domain"])["_u"].rank(method="first")
        keep = keep[keep["_rank"] <= k].drop(columns="_rank")
    return keep.drop(columns="_u").reset_index(drop=True)

def load_raid_hf(cfg, max_rows=3_000_000):
    """Fallback: stream from the Hugging Face hub (row order may be grouped, so we read widely)."""
    from datasets import load_dataset
    ds = load_dataset("liamdugan/raid", "raid", split="train", streaming=True)
    ds = ds.shuffle(seed=cfg["seed"], buffer_size=100_000)
    k_machine = math.ceil(cfg["per_model_n"] / 8); k_human = math.ceil(cfg["human_n"] / 8)
    counts, rows = {}, []
    for i, ex in enumerate(ds):
        if i >= max_rows: break
        if (ex.get("attack") or "none") != "none" or not ex.get("generation"): continue
        if len(ex["generation"]) < cfg["min_chars"]: continue
        key = (ex["model"], ex["domain"]); cap = k_human if ex["model"] == "human" else k_machine
        if counts.get(key, 0) >= cap: continue
        counts[key] = counts.get(key, 0) + 1
        rows.append({c: ex.get(c) for c in USECOLS})
    return pd.DataFrame(rows)

def prepare(df, cfg):
    df = df.copy()
    text = df["generation"].astype(str)
    if cfg.get("flatten_layout", True):
        # Remove the paragraph-layout shortcut: all whitespace runs (incl. newlines) -> one space
        text = text.str.replace(r"\s+", " ", regex=True).str.strip()
    df["text"] = text.str.slice(0, cfg["max_chars"])
    df["label"] = (df["model"] != "human").astype(int)
    df["family"] = df["model"].map(FAMILY).fillna("other")
    # stratified split on generator (human stratified by domain)
    rng = np.random.default_rng(cfg["seed"])
    strat = np.where(df["label"] == 1, df["model"], "human_" + df["domain"].astype(str))
    split = np.empty(len(df), dtype=object)
    tr, va, _ = cfg["split"]
    for s in np.unique(strat):
        idx = np.where(strat == s)[0]; rng.shuffle(idx)
        n_tr, n_va = int(len(idx) * tr), int(len(idx) * va)
        split[idx[:n_tr]] = "train"; split[idx[n_tr:n_tr + n_va]] = "val"; split[idx[n_tr + n_va:]] = "test"
    df["split"] = split
    return df.drop(columns=["generation"]).reset_index(drop=True)

sample_path = os.path.join(CFG["out_dir"], "sample_ids.csv")
try:
    raw = stratified_sample_csv(download_raid(CFG), CFG)
except Exception as e:
    print("CSV route failed:", e, "→ using Hugging Face streaming")
    raw = load_raid_hf(CFG)
DF = prepare(raw, CFG)
DF[["id", "model", "domain", "split"]].to_csv(sample_path, index=False)   # for reproducibility
print(DF.groupby(["split", "label"]).size().unstack())
print(DF[DF.split == "train"]["model"].value_counts())

## 2. Attacks and the normalization defense
Attacks are applied **only to machine-generated test text** (as in RAID); human text stays clean.
The defense is a cheap preprocessing step: Unicode NFKC, zero-width removal, Cyrillic→Latin
homoglyph mapping and whitespace collapsing.

In [ ]:
HOMO = {'a': 'а', 'e': 'е', 'o': 'о', 'c': 'с', 'p': 'р', 'x': 'х', 'i': 'і', 'y': 'у',
        'A': 'А', 'E': 'Е', 'O': 'О', 'C': 'С', 'P': 'Р', 'X': 'Х', 'B': 'В', 'H': 'Н',
        'K': 'К', 'M': 'М', 'T': 'Т'}
REV_HOMO = {v: k for k, v in HOMO.items()}
ZW = ['​', '‌', '‍', '⁠', '﻿']
ZW_RE = re.compile("[" + "".join(ZW) + "]")

def normalize(t: str) -> str:
    t = unicodedata.normalize("NFKC", t)
    t = ZW_RE.sub("", t)
    t = "".join(REV_HOMO.get(ch, ch) for ch in t)
    t = re.sub(r"[ \t]+", " ", t)
    return t

def atk_homoglyph(t, r, rate=0.15):
    return "".join(HOMO[c] if c in HOMO and r.random() < rate else c for c in t)

def atk_zero_width(t, r, rate=0.10):
    out = []
    for c in t:
        out.append(c)
        if c.isalpha() and r.random() < rate:
            out.append(r.choice(ZW))
    return "".join(out)

def atk_whitespace(t, r, rate=0.20):
    return "".join(("  " if (c == " " and r.random() < rate) else c) for c in t)

def atk_typos(t, r, rate=0.08):
    words = t.split(" ")
    for i, w in enumerate(words):
        if len(w) > 3 and w.isalpha() and r.random() < rate:
            j = r.randrange(1, len(w) - 2)
            words[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return " ".join(words)

def atk_article_deletion(t, r):
    return re.sub(r"\b(?:a|an|the)\s+", "", t, flags=re.IGNORECASE)

def atk_case_flip(t, r, rate=0.05):
    return "".join(c.swapcase() if c.isalpha() and r.random() < rate else c for c in t)

_WN = None
def atk_synonym(t, r, rate=0.10):
    global _WN
    if _WN is None:
        import nltk
        for pkg in ("wordnet", "omw-1.4"):
            try: nltk.data.find(f"corpora/{pkg}")
            except LookupError: nltk.download(pkg, quiet=True)
        from nltk.corpus import wordnet as wn; _WN = wn
    words = t.split(" ")
    for i, w in enumerate(words):
        if w.isalpha() and len(w) > 3 and r.random() < rate:
            syns = {l.name().replace("_", " ") for s in _WN.synsets(w.lower())[:3] for l in s.lemmas()}
            syns.discard(w.lower())
            if syns:
                s = r.choice(sorted(syns))
                words[i] = s.capitalize() if w[0].isupper() else s
    return " ".join(words)

def atk_combined(t, r):
    return atk_typos(atk_zero_width(atk_homoglyph(t, r), r), r)

ATTACKS = {"homoglyph": atk_homoglyph, "zero_width": atk_zero_width, "whitespace": atk_whitespace,
           "typos": atk_typos, "article_deletion": atk_article_deletion, "case_flip": atk_case_flip,
           "synonym": atk_synonym, "combined": atk_combined}

def apply_attack(texts, name, seed):
    r = random.Random(seed)
    return [ATTACKS[name](t, r) for t in texts]

def paraphrase_attack(texts, batch=16):
    """Optional: sentence-level paraphrase with a public T5 paraphraser."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    name = "humarin/chatgpt_paraphraser_on_T5_base"
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
    out = []
    for t in texts:
        sents = re.split(r"(?<=[.!?])\s+", t)[:12]
        para = []
        for i in range(0, len(sents), batch):
            enc = tok(["paraphrase: " + s for s in sents[i:i + batch]], return_tensors="pt",
                      padding=True, truncation=True, max_length=128).to(DEVICE)
            with torch.no_grad():
                g = mdl.generate(**enc, max_length=128, num_beams=4)
            para += tok.batch_decode(g, skip_special_tokens=True)
        out.append(" ".join(para))
    return out

## 3. Features: stylometry (D2) and small-LM perplexity statistics (D3)

In [ ]:
STOP = set("""a about above after again against all am an and any are as at be because been before being
below between both but by can could did do does doing down during each few for from further had has have
having he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not now of off on once only or other our ours ourselves out over own same she should so some
such than that the their theirs them themselves then there these they this those through to too under
until up very was we were what when where which while who whom why will with would you your yours yourself""".split())
STYLO_COLS = ["n_chars", "n_words", "avg_word_len", "std_word_len", "n_sents", "avg_sent_len",
              "std_sent_len", "burstiness", "ttr", "hapax_ratio", "punct_ratio", "comma_per_word",
              "stop_ratio", "upper_ratio", "digit_ratio", "newline_ratio", "nonascii_ratio",
              "q_excl_ratio", "repeat_bigram_ratio", "avg_para_len"]

def stylometric(t: str):
    n = max(len(t), 1)
    words = re.findall(r"[A-Za-zЀ-ӿ']+", t)
    nw = max(len(words), 1)
    wl = np.array([len(w) for w in words]) if words else np.array([0])
    sents = [s for s in re.split(r"(?<=[.!?])\s+|\n+", t) if s.strip()]
    sl = np.array([len(s.split()) for s in sents]) if sents else np.array([0])
    lw = [w.lower() for w in words]
    counts = pd.Series(lw).value_counts() if lw else pd.Series(dtype=int)
    bigrams = list(zip(lw, lw[1:]))
    paras = [p for p in t.split("\n\n") if p.strip()]
    return [len(t), len(words), wl.mean(), wl.std(), len(sents), sl.mean(), sl.std(),
            sl.std() / (sl.mean() + 1e-6),
            len(counts) / nw, (counts == 1).sum() / nw,
            sum(c in ".,;:!?\"'()-" for c in t) / n, t.count(",") / nw,
            sum(w in STOP for w in lw) / nw,
            sum(c.isupper() for c in t) / n, sum(c.isdigit() for c in t) / n, t.count("\n") / n,
            sum(ord(c) > 127 for c in t) / n, (t.count("?") + t.count("!")) / max(len(sents), 1),
            1 - len(set(bigrams)) / max(len(bigrams), 1),
            np.mean([len(p.split()) for p in paras]) if paras else 0.0]

def stylo_matrix(texts):
    return np.asarray([stylometric(t) for t in texts], dtype=np.float32)

PPL_COLS = ["mean_logp", "std_logp", "mean_entropy", "std_entropy", "mean_logrank", "top1_frac",
            "top10_frac", "logp_entropy_ratio", "min_logp_p10"]

class PPLScorer:
    def __init__(self, name, max_len):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(name)
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        self.tok.padding_side = "right"
        dtype = torch.float16 if DEVICE == "cuda" else torch.float32
        self.m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(DEVICE).eval()
        self.max_len = max_len

    def features(self, texts, batch=16):
        torch = self.torch; feats = []
        for i in range(0, len(texts), batch):
            enc = self.tok(texts[i:i + batch], return_tensors="pt", padding=True, truncation=True,
                           max_length=self.max_len).to(DEVICE)
            with torch.no_grad():
                logits = self.m(**enc).logits[:, :-1].float()
            tgt = enc["input_ids"][:, 1:]; mask = enc["attention_mask"][:, 1:].bool()
            logp_all = torch.log_softmax(logits, -1)
            logp = logp_all.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
            ent = -(logp_all.exp() * logp_all).sum(-1)
            tgt_logit = logits.gather(-1, tgt.unsqueeze(-1))
            rank = (logits > tgt_logit).sum(-1).float() + 1
            for b in range(tgt.shape[0]):
                mk = mask[b]
                if mk.sum() < 2:
                    feats.append([0.0] * len(PPL_COLS)); continue
                lp, en, rk = logp[b][mk].cpu().numpy(), ent[b][mk].cpu().numpy(), rank[b][mk].cpu().numpy()
                feats.append([lp.mean(), lp.std(), en.mean(), en.std(), np.log(rk).mean(),
                              (rk == 1).mean(), (rk <= 10).mean(), lp.mean() / (en.mean() + 1e-6),
                              np.percentile(lp, 10)])
            del logits, logp_all, ent
        return np.asarray(feats, dtype=np.float32)

## 4. Detectors

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

def metrics(y, p, thr=0.5):
    y, p = np.asarray(y), np.asarray(p)
    fpr, tpr, _ = roc_curve(y, p)
    return dict(acc=accuracy_score(y, p >= thr), f1=f1_score(y, p >= thr),
                auroc=roc_auc_score(y, p), tpr_at_5fpr=float(np.interp(0.05, fpr, tpr)))

def tpr_at_fixed_threshold(p_machine, thr):
    return float((np.asarray(p_machine) >= thr).mean())

def threshold_for_fpr(p_human_val, fpr=0.05):
    return float(np.quantile(np.asarray(p_human_val), 1 - fpr))

class TfidfLR:
    name = "D1 TF-IDF(char)+LR"
    def fit(self, X, y):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=200_000,
                                   sublinear_tf=True, min_df=2)
        self.clf = LogisticRegression(C=4.0, max_iter=3000, class_weight="balanced")
        self.clf.fit(self.vec.fit_transform(X), y); return self
    def predict_proba(self, X): return self.clf.predict_proba(self.vec.transform(X))[:, 1]
    def n_params(self): return self.clf.coef_.size

class FeatClf:
    """Classifier over a precomputed feature matrix (stylometric or perplexity)."""
    def __init__(self, name, kind):
        self.name, self.kind = name, kind
    def fit(self, F, y):
        if self.kind == "gbm":
            self.clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                                      class_weight="balanced", random_state=CFG["seed"])
        else:
            self.clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
        self.clf.fit(F, y); return self
    def predict_proba(self, F): return self.clf.predict_proba(F)[:, 1]
    def n_params(self):
        return None if self.kind == "gbm" else self.clf[-1].coef_.size + 1

class TransformerClf:
    name = "D4 DistilRoBERTa (fine-tuned)"
    def __init__(self, cfg, epochs=None):
        self.cfg = cfg; self.epochs = epochs or cfg["epochs"]
    def fit(self, X, y):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
        set_seed(self.cfg["seed"])
        self.tok = AutoTokenizer.from_pretrained(self.cfg["tf_model"])
        self.m = AutoModelForSequenceClassification.from_pretrained(self.cfg["tf_model"], num_labels=2).to(DEVICE)
        y = np.asarray(y); w = torch.tensor([len(y) / (2 * (y == 0).sum()), len(y) / (2 * (y == 1).sum())],
                                            dtype=torch.float, device=DEVICE)
        opt = torch.optim.AdamW(self.m.parameters(), lr=self.cfg["lr"], weight_decay=0.01)
        bs = self.cfg["batch"]; steps = self.epochs * math.ceil(len(X) / bs)
        sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
        scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
        lossf = torch.nn.CrossEntropyLoss(weight=w)
        X = list(X)
        for ep in range(self.epochs):
            self.m.train(); order = np.random.permutation(len(X)); tot = 0.0
            for i in range(0, len(X), bs):
                idx = order[i:i + bs]
                enc = self.tok([X[j] for j in idx], return_tensors="pt", padding=True, truncation=True,
                               max_length=self.cfg["max_len"]).to(DEVICE)
                with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                    loss = lossf(self.m(**enc).logits, torch.tensor(y[idx], device=DEVICE))
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(self.m.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
            print(f"  epoch {ep + 1}/{self.epochs} loss={tot / math.ceil(len(X) / bs):.4f}")
        return self
    def predict_proba(self, X, bs=64):
        import torch
        self.m.eval(); out = []; X = list(X)
        for i in range(0, len(X), bs):
            enc = self.tok(X[i:i + bs], return_tensors="pt", padding=True, truncation=True,
                           max_length=self.cfg["max_len"]).to(DEVICE)
            with torch.no_grad(), torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                out.append(torch.softmax(self.m(**enc).logits.float(), -1)[:, 1].cpu().numpy())
        return np.concatenate(out)
    def n_params(self): return sum(p.numel() for p in self.m.parameters())

class FusionLR:
    """Late fusion: logistic meta-learner over base-detector logits, trained on the validation split."""
    name = "Proposed: Norm + Fusion"
    @staticmethod
    def _z(P): P = np.clip(P, 1e-5, 1 - 1e-5); return np.log(P / (1 - P))
    def fit(self, P_val, y_val):
        self.clf = LogisticRegression(C=1.0, max_iter=1000).fit(self._z(P_val), y_val); return self
    def predict_proba(self, P): return self.clf.predict_proba(self._z(P))[:, 1]

## 5. Training pipeline (shared by E1–E3)

In [ ]:
BASE_NAMES = ["D1 TF-IDF(char)+LR", "D2 Stylometry+GBM", "D3 Perplexity(Pythia-160M)+LR",
              "D4 DistilRoBERTa (fine-tuned)"]

def feature_cache(texts, scorer):
    return stylo_matrix(texts), scorer.features(texts, CFG["ppl_batch"])

class Suite:
    """Trains D1–D4 on a train split, the fusion on a val split, and scores any text set."""
    def __init__(self, cfg, scorer, tf_epochs=None):
        self.cfg, self.scorer, self.tf_epochs = cfg, scorer, tf_epochs
    def fit(self, tr, va, Ftr, Fva):
        ytr, yva = tr["label"].values, va["label"].values
        self.d1 = TfidfLR().fit(tr["text_n"].tolist(), ytr)
        self.d2 = FeatClf(BASE_NAMES[1], "gbm").fit(Ftr["stylo"], ytr)
        self.d3 = FeatClf(BASE_NAMES[2], "lr").fit(Ftr["ppl"], ytr)
        print(" training transformer…"); self.d4 = TransformerClf(self.cfg, self.tf_epochs).fit(tr["text_n"].tolist(), ytr)
        Pva = self.base_probs(va["text_n"].tolist(), Fva)
        self.fusion = FusionLR().fit(Pva, yva)
        self.thr = {n: threshold_for_fpr(Pva[yva == 0, i]) for i, n in enumerate(BASE_NAMES)}
        self.thr[FusionLR.name] = threshold_for_fpr(self.fusion.predict_proba(Pva)[yva == 0])
        return self
    def base_probs(self, texts, F):
        return np.column_stack([self.d1.predict_proba(texts), self.d2.predict_proba(F["stylo"]),
                                self.d3.predict_proba(F["ppl"]), self.d4.predict_proba(texts)])
    def all_probs(self, texts, F=None):
        if F is None:
            s, p = feature_cache(texts, self.scorer); F = {"stylo": s, "ppl": p}
        P = self.base_probs(texts, F)
        return {**{n: P[:, i] for i, n in enumerate(BASE_NAMES)}, FusionLR.name: self.fusion.predict_proba(P)}

def subset_feats(F, mask):
    return {k: v[mask] for k, v in F.items()}

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
DF["text_n"] = [normalize(t) for t in DF["text"]]
t0 = time.time()
SCORER = PPLScorer(CFG["ppl_model"], CFG["max_len"])
S_ALL, P_ALL = feature_cache(DF["text_n"].tolist(), SCORER)
FEATS = {"stylo": S_ALL, "ppl": P_ALL}
print(f"Features for {len(DF)} docs in {time.time() - t0:.0f}s")
pd.DataFrame(np.hstack([S_ALL, P_ALL]), columns=STYLO_COLS + PPL_COLS).assign(
    label=DF["label"].values, model=DF["model"].values).to_csv(
    os.path.join(CFG["out_dir"], "features.csv"), index=False)

## E1. In-distribution detection

In [ ]:
def save_table(df, name, caption):
    df.to_csv(os.path.join(CFG["out_dir"], f"{name}.csv"))
    with open(os.path.join(CFG["out_dir"], f"{name}.tex"), "w") as f:
        f.write(df.to_latex(float_format="%.3f", caption=caption, label=f"tab:{name}"))

m_tr, m_va, m_te = (DF.split == "train").values, (DF.split == "val").values, (DF.split == "test").values
TR, VA, TE = DF[m_tr], DF[m_va], DF[m_te]
SUITE = Suite(CFG, SCORER).fit(TR, VA, subset_feats(FEATS, m_tr), subset_feats(FEATS, m_va))
PT = SUITE.all_probs(TE["text_n"].tolist(), subset_feats(FEATS, m_te))
T1 = pd.DataFrame({n: metrics(TE["label"], p) for n, p in PT.items()}).T
save_table(T1, "table1_in_distribution", "In-distribution detection on RAID (test split).")
display(T1) if "display" in globals() else print(T1)

# per-generator and per-domain TPR at the val-calibrated 5% FPR threshold
rows = []
for n, p in PT.items():
    thr = SUITE.thr[n]
    for g, idx in TE[TE.label == 1].groupby("model").groups.items():
        rows.append(dict(detector=n, group=g, kind="generator",
                         tpr=tpr_at_fixed_threshold(p[TE.index.get_indexer(idx)], thr)))
    for d, idx in TE[TE.label == 1].groupby("domain").groups.items():
        rows.append(dict(detector=n, group=d, kind="domain",
                         tpr=tpr_at_fixed_threshold(p[TE.index.get_indexer(idx)], thr)))
BREAK = pd.DataFrame(rows)
save_table(BREAK.pivot_table(index="detector", columns="group", values="tpr"),
           "table1b_breakdown", "TPR at 5% FPR by generator and domain.")

## E2. Robustness to attacks, with and without the normalization defense

In [ ]:
rng = np.random.default_rng(CFG["seed"])
te_m = TE[TE.label == 1]
te_m = te_m.iloc[rng.choice(len(te_m), min(CFG["attack_n"], len(te_m)), replace=False)]
clean_texts = te_m["text"].tolist()
p_h = {n: p[(TE.label == 0).values] for n, p in PT.items()}   # clean human test scores
attack_sets = {"none": clean_texts}
for i, a in enumerate(ATTACKS):
    attack_sets[a] = apply_attack(clean_texts, a, CFG["seed"] + i)
if CFG["run_paraphrase"]:
    attack_sets["paraphrase"] = paraphrase_attack(clean_texts[:CFG["paraphrase_n"]])
rows = []
for a, texts in attack_sets.items():
    for defended in (False, True):
        if a == "none" and not defended: continue
        inp = [normalize(t) for t in texts] if defended else texts
        P = SUITE.all_probs(inp)
        for n, pm in P.items():
            y = np.r_[np.zeros(len(p_h[n])), np.ones(len(pm))]
            auc = roc_auc_score(y, np.r_[p_h[n], pm])
            rows.append(dict(attack=a, defense="normalized" if defended else "raw", detector=n,
                             tpr_at_5fpr=tpr_at_fixed_threshold(pm, SUITE.thr[n]), auroc=auc))
    print("done:", a)
ROB = pd.DataFrame(rows)
ROB.to_csv(os.path.join(CFG["out_dir"], "table2_robustness_long.csv"), index=False)
T2 = ROB.pivot_table(index=["detector", "defense"], columns="attack", values="tpr_at_5fpr")
save_table(T2, "table2_robustness", "TPR at 5% FPR under attacks (raw vs. normalized input).")
print(T2.round(3))

## E3. Generalisation to an unseen LLM family (leave-one-family-out)

In [ ]:
if CFG["run_logo"]:
    rows = []
    for fam in sorted(f for f in DF.family.unique() if f != "human"):
        print(f"LOGO fold: hold out {fam}")
        keep_tr = m_tr & ((DF.family != fam) | (DF.label == 0)).values
        keep_va = m_va & ((DF.family != fam) | (DF.label == 0)).values
        keep_te = m_te & ((DF.family == fam) | (DF.label == 0)).values
        s = Suite(CFG, SCORER, tf_epochs=CFG["logo_epochs"]).fit(
            DF[keep_tr], DF[keep_va], subset_feats(FEATS, keep_tr), subset_feats(FEATS, keep_va))
        te = DF[keep_te]; P = s.all_probs(te["text_n"].tolist(), subset_feats(FEATS, keep_te))
        for n, p in P.items():
            rows.append(dict(held_out=fam, detector=n, **metrics(te["label"], p),
                             tpr_fixed=tpr_at_fixed_threshold(p[te.label.values == 1], s.thr[n])))
        del s; gc.collect(); torch.cuda.empty_cache()
    LOGO = pd.DataFrame(rows)
    LOGO.to_csv(os.path.join(CFG["out_dir"], "table3_logo_long.csv"), index=False)
    T3 = LOGO.pivot_table(index="detector", columns="held_out", values="auroc")
    T3["mean"] = T3.mean(axis=1)
    save_table(T3, "table3_logo", "AUROC when the test LLM family is unseen in training.")
    print(T3.round(3))

## E4. Efficiency

In [ ]:
def time_it(fn, n_rep=1):
    t = time.time()
    for _ in range(n_rep): fn()
    return (time.time() - t) / n_rep

sample = TE["text_n"].tolist()[:500]; n = len(sample)
t_style = time_it(lambda: stylo_matrix(sample))
t_ppl = time_it(lambda: SCORER.features(sample, CFG["ppl_batch"]))
Fs = {"stylo": stylo_matrix(sample), "ppl": SCORER.features(sample, CFG["ppl_batch"])}
t_d1 = time_it(lambda: SUITE.d1.predict_proba(sample))
t_d2 = time_it(lambda: SUITE.d2.predict_proba(Fs["stylo"])) + t_style
t_d3 = time_it(lambda: SUITE.d3.predict_proba(Fs["ppl"])) + t_ppl
t_d4 = time_it(lambda: SUITE.d4.predict_proba(sample))
ppl_params = sum(p.numel() for p in SCORER.m.parameters())
T4 = pd.DataFrame({
    "params": [SUITE.d1.n_params(), None, ppl_params, SUITE.d4.n_params(),
               SUITE.d1.n_params() + ppl_params + SUITE.d4.n_params()],
    "ms_per_doc": [1000 * t / n for t in (t_d1, t_d2, t_d3, t_d4, t_d1 + t_d2 + t_d3 + t_d4)],
}, index=BASE_NAMES + [FusionLR.name])
T4["device"] = DEVICE
save_table(T4, "table4_efficiency", "Model size and inference latency per document.")
print(T4)

## Figures (300 dpi, ready for the manuscript)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
plt.rcParams.update({"font.size": 10, "font.family": "serif"})
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(CFG["out_dir"], name), dpi=300); plt.show()

# Fig 1 – in-distribution
ax = T1[["auroc", "tpr_at_5fpr", "f1"]].plot.barh(figsize=(7, 3.6), width=0.8)
ax.set_xlim(0.5, 1.0); ax.set_xlabel("Score"); ax.legend(["AUROC", "TPR@5%FPR", "F1"], loc="lower right")
savefig("fig1_in_distribution.png")

# Fig 2 – robustness heatmaps (raw vs normalized)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, d in zip(axes, ["raw", "normalized"]):
    h = ROB[ROB.defense == d].pivot_table(index="detector", columns="attack", values="tpr_at_5fpr")
    sns.heatmap(h, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax, cbar=d == "normalized")
    ax.set_title(f"TPR@5%FPR – {d} input"); ax.set_xlabel("")
savefig("fig2_robustness.png")

# Fig 3 – per-generator TPR
h = BREAK[BREAK.kind == "generator"].pivot_table(index="detector", columns="group", values="tpr")
plt.figure(figsize=(10, 3.4)); sns.heatmap(h, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1)
plt.xlabel("Generator"); plt.ylabel(""); savefig("fig3_per_generator.png")

# Fig 4 – LOGO
if CFG["run_logo"]:
    T3.drop(columns="mean").T.plot.bar(figsize=(8, 3.4), width=0.85)
    plt.ylim(0.4, 1.0); plt.ylabel("AUROC"); plt.xlabel("Held-out LLM family")
    plt.legend(fontsize=7, ncol=2); savefig("fig4_logo.png")

# Fig 5 – feature distributions (human vs machine) for the most discriminative cues
feat_df = pd.read_csv(os.path.join(CFG["out_dir"], "features.csv"))
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8))
for ax, c in zip(axes, ["mean_logp", "mean_logrank", "burstiness", "ttr"]):
    sns.kdeplot(data=feat_df, x=c, hue="label", common_norm=False, fill=True, ax=ax, legend=ax is axes[0])
savefig("fig5_feature_distributions.png")

json.dump(CFG, open(os.path.join(CFG["out_dir"], "config.json"), "w"), indent=2)
ZIP = (f"/content/results_{CFG['run_tag']}.zip" if os.path.isdir("/content")
       else f"results_{CFG['run_tag']}.zip")
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(CFG["out_dir"]):
        z.write(os.path.join(CFG["out_dir"], f), f)
print("Saved", ZIP)
try:
    from google.colab import files; files.download(ZIP)
except Exception:
    pass